# CareLoop Clinics — Part 3
Machine Learning: predict no-shows and choose the model honestly.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

df = pd.read_csv("appointments_clean.csv")


## Task 3.1 — features, split and baseline

In [ ]:
X = df[[
    "age","distance_km","fee_inr","lead_days","prior_no_shows",
    "reminder_sent","booking_channel","visit_type"
]].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes":1, "No":0})
X = pd.get_dummies(X, columns=["booking_channel","visit_type"])
y = df["no_show"]

print("Feature columns:", X.columns.tolist())
print("X shape:", X.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("Test no-shows:", int(y_test.sum()))
print("Baseline accuracy:", (y_test == 0).mean())


## Task 3.2 — logistic regression

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)

pred = model.predict(X_test_s)
prob = model.predict_proba(X_test_s)[:,1]

print("Accuracy:", accuracy_score(y_test, pred))
print("Confusion matrix:\n", confusion_matrix(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("Recall:", recall_score(y_test, pred))
print("F1:", f1_score(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, prob))

for name, coef in zip(X.columns, model.coef_[0]):
    print(name, coef)

reminder_coef = model.coef_[0][list(X.columns).index("reminder_sent")]
reminder_sd = scaler.scale_[list(X.columns).index("reminder_sent")]
print("exp(scaled coefficient):", np.exp(reminder_coef))
print("exp(coef / SD), No -> Yes:", np.exp(reminder_coef / reminder_sd))


## Task 3.3 — decision trees

In [ ]:
full_tree = DecisionTreeClassifier(random_state=42)
depth3_tree = DecisionTreeClassifier(max_depth=3, random_state=42)

full_tree.fit(X_train, y_train)
depth3_tree.fit(X_train, y_train)

print("Full tree train/test:", full_tree.score(X_train,y_train), full_tree.score(X_test,y_test))
print("Full tree depth:", full_tree.get_depth())
print("Depth-3 train/test:", depth3_tree.score(X_train,y_train), depth3_tree.score(X_test,y_test))


## Task 3.4 — random forest and comparison

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_prob = rf.predict_proba(X_test)[:,1]
rf_pred = rf.predict(X_test)

print("RF accuracy:", accuracy_score(y_test, rf_pred))
print("RF recall:", recall_score(y_test, rf_pred))
print("RF ROC-AUC:", roc_auc_score(y_test, rf_prob))

importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 3 feature importances:\n", importance.head(3))

comparison = pd.DataFrame([
    ["Logistic regression", accuracy_score(y_test,pred), recall_score(y_test,pred), roc_auc_score(y_test,prob)],
    ["Depth-3 tree", depth3_tree.score(X_test,y_test), recall_score(y_test,depth3_tree.predict(X_test)), roc_auc_score(y_test,depth3_tree.predict_proba(X_test)[:,1])],
    ["Random forest", accuracy_score(y_test,rf_pred), recall_score(y_test,rf_pred), roc_auc_score(y_test,rf_prob)]
], columns=["model","accuracy","recall","roc_auc"])
print(comparison)


## Task 3.5 — tune, choose, threshold, test once

In [ ]:
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {"n_estimators":[100,200], "max_depth":[3,5,8]},
    cv=5, scoring="roc_auc"
)
grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print("Best CV ROC-AUC:", grid.best_score_)

lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

cv_scores = {
    "logistic_regression": cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "depth_3_tree": cross_val_score(depth3_tree, X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "random_forest": cross_val_score(rf, X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "tuned_random_forest": grid.best_score_
}
print(cv_scores)

winner = lr_pipe if cv_scores["logistic_regression"] >= max(cv_scores.values()) else grid.best_estimator_

oof = cross_val_predict(winner, X_train, y_train, cv=5, method="predict_proba")[:,1]
for threshold in [0.5, 0.35]:
    p = (oof >= threshold).astype(int)
    print(
        threshold,
        "precision=", precision_score(y_train,p),
        "recall=", recall_score(y_train,p),
        "f1=", f1_score(y_train,p),
        "flagged=", int(p.sum())
    )

chosen_threshold = 0.35
winner.fit(X_train, y_train)
probs = winner.predict_proba(X_test)[:,1]
flags = (probs >= chosen_threshold).astype(int)

print("Final test precision:", precision_score(y_test,flags))
print("Final test recall:", recall_score(y_test,flags))
print("Final flagged:", int(flags.sum()))
print("Final ROC-AUC:", roc_auc_score(y_test,probs))

test_risk_scores = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": probs,
    "flag": flags
})
test_risk_scores.to_csv("test_risk_scores.csv", index=False)
